# KAN Picbreeder: Comprehensive Analysis

Comparing KAN variants (spline KAN, SwarmKAN, MemeticKAN) against standard MLPs for CPPN-based image generation. This notebook answers six research questions about performance, learned representations, scalability, and transferable design insights.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from analysis.benchmark import (
    run_full_benchmark, load_target_image, GENOME_CONFIGS,
    get_pretrained_reference
)
from analysis.comparison import mse, ssim, feature_cosine_similarity
from analysis.spline_inspector import extract_spline_curve, fit_known_function, analyze_all_edges
from analysis.text_prototype import SequenceKAN, train_sequence_kan, make_test_signals

plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 11

%matplotlib inline

GENOME = 'skull'  # Change to 'butterfly' or 'apple' to analyze other genomes

## 1. Method Comparison: Who Wins?

We train 4 methods from scratch on the same target image with 3 random seeds, plus compare against pre-trained picbreeder and SGD genomes as reference bounds.

**Methods:**
- **MLP+SGD**: Standard CPPN with fixed activations (identity, gaussian, sin, sigmoid), trained with normalized-gradient SGD
- **KAN+SGD**: Same architecture but with learnable spline activations, trained with SGD
- **SwarmKAN**: KAN + Particle Swarm Optimization on spline coefficients (PSO every 5 SGD steps)
- **MemeticKAN**: KAN + Natural Evolution Strategy (antithetic sampling) + SGD local refinement
- **Picbreeder reference**: Pre-evolved via interactive evolution (upper bound for interpretability)
- **SGD pre-trained**: Pre-trained with standard SGD from the FER paper

In [ ]:
# Run full benchmark (this takes a few minutes)
results, target_img = run_full_benchmark(GENOME, n_iters=5000, n_seeds=3, n_generations=50)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Convergence curves ---
ax = axes[0]
colors = {'mlp_sgd': '#1f77b4', 'kan_sgd': '#ff7f0e', 'swarm_kan': '#2ca02c', 'memetic_kan': '#d62728'}
labels = {'mlp_sgd': 'MLP+SGD', 'kan_sgd': 'KAN+SGD', 'swarm_kan': 'SwarmKAN', 'memetic_kan': 'MemeticKAN'}

for method, runs in results.items():
    if method in ('picbreeder', 'sgd_pretrained'):
        continue
    all_losses = np.array([r['losses'] for r in runs])
    mean_loss = all_losses.mean(axis=0)
    std_loss = all_losses.std(axis=0)

    if method == 'memetic_kan':
        # Memetic losses are per-generation, stretch to match iteration scale
        total_iters = runs[0]['total_iters']
        x = np.linspace(0, total_iters, len(mean_loss))
    else:
        x = np.arange(len(mean_loss))

    ax.plot(x, mean_loss, color=colors[method], label=labels[method], linewidth=1.5)
    ax.fill_between(x, mean_loss - std_loss, mean_loss + std_loss, alpha=0.2, color=colors[method])

ax.set_yscale('log')
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE (log scale)')
ax.set_title(f'Convergence Curves \u2014 {GENOME.title()}')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Final MSE bar chart ---
ax = axes[1]
method_names = []
final_mses = []
final_stds = []

for method in ['mlp_sgd', 'kan_sgd', 'swarm_kan', 'memetic_kan']:
    method_names.append(labels[method])
    finals = [r['losses'][-1] for r in results[method]]
    final_mses.append(np.mean(finals))
    final_stds.append(np.std(finals))

# Add pre-trained references
for ref_name, ref_label in [('picbreeder', 'Picbreeder'), ('sgd_pretrained', 'SGD-pretrained')]:
    ref_img = results[ref_name][0]['final_img']
    ref_mse = mse(ref_img, target_img)
    method_names.append(ref_label)
    final_mses.append(ref_mse)
    final_stds.append(0)

bars = ax.bar(method_names, final_mses, yerr=final_stds, capsize=4,
              color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b'])
ax.set_ylabel('Final MSE')
ax.set_title(f'Final Image Quality \u2014 {GENOME.title()}')
ax.tick_params(axis='x', rotation=30)

# --- Wall-clock time ---
ax = axes[2]
time_names = []
times = []
time_stds = []

for method in ['mlp_sgd', 'kan_sgd', 'swarm_kan', 'memetic_kan']:
    time_names.append(labels[method])
    ts = [r['wall_time'] for r in results[method]]
    times.append(np.mean(ts))
    time_stds.append(np.std(ts))

ax.bar(time_names, times, yerr=time_stds, capsize=4,
       color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'])
ax.set_ylabel('Wall-clock Time (seconds)')
ax.set_title(f'Training Time \u2014 {GENOME.title()}')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# SSIM comparison
fig, axes = plt.subplots(1, 6, figsize=(20, 4))

imgs_to_show = [
    ('Target', target_img),
]
for method, label in [('mlp_sgd', 'MLP+SGD'), ('kan_sgd', 'KAN+SGD'),
                       ('swarm_kan', 'SwarmKAN'), ('memetic_kan', 'MemeticKAN'),
                       ('picbreeder', 'Picbreeder')]:
    imgs_to_show.append((label, results[method][0]['final_img']))

for i, (label, img) in enumerate(imgs_to_show):
    ax = axes[i]
    img_np = img.detach().cpu().numpy() if isinstance(img, torch.Tensor) else img
    ax.imshow(img_np.clip(0, 1))
    ax.set_title(label, fontsize=10)
    if i > 0:
        s = ssim(target_img, img)
        m = mse(target_img, img)
        ax.set_xlabel(f'MSE={m:.4f}\nSSIM={s:.3f}', fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle(f'Image Comparison \u2014 {GENOME.title()}', fontsize=14)
plt.tight_layout()
plt.show()

## 2. How Close Are the Learned Functions?

Each KAN edge has a learnable spline activation. We extract these curves and compare them against the CPPN's actual activation functions (gaussian, sin, sigmoid, identity, tanh).

**Method:** For each edge, we:
1. Evaluate the spline at 1000 points in [-3, 3]
2. Fit `a * f(b*x + c) + d` for each known function f
3. Pick the best fit by L2 distance
4. Build a heatmap: what did each edge learn?

In [ ]:
# Get the best KAN model from seed 0
kan_model = results['kan_sgd'][0]['model']

# Analyze top-20 most active edges
top_edges = analyze_all_edges(kan_model, top_k=20)

# --- Grid of learned spline vs. best match ---
n_show = min(12, len(top_edges))
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for i, edge in enumerate(top_edges[:n_show]):
    ax = axes[i // 4, i % 4]

    ax.plot(edge['raw_inputs'], edge['spline_values'], 'b-', linewidth=2, label='Learned spline')

    match = edge['best_match']
    if match['fitted_curve'] is not None:
        ax.plot(edge['raw_inputs'], match['fitted_curve'], 'r--', linewidth=1.5,
                label=f"Best fit: {match['name']}")

    ax.set_title(f"L{edge['layer_idx']} [{edge['in_idx']},{edge['out_idx']}]\n"
                 f"\u2248 {match['name']} (L2={match['l2_distance']:.3f})",
                 fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-3, 3)

plt.suptitle(f'Learned Spline Activations vs. Known Functions \u2014 {GENOME.title()} KAN', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Build function-type heatmap across all layers
function_counts = {}
all_edges_full = analyze_all_edges(kan_model, top_k=999999)  # Get all edges

for edge in all_edges_full:
    layer = edge['layer_idx']
    fn_name = edge['best_match']['name']
    key = (layer, fn_name)
    function_counts[key] = function_counts.get(key, 0) + 1

# Create heatmap matrix
n_layers_total = len(kan_model.layers)
fn_names = sorted(set(e['best_match']['name'] for e in all_edges_full))

heatmap = np.zeros((n_layers_total, len(fn_names)))
for (layer, fn), count in function_counts.items():
    col = fn_names.index(fn)
    heatmap[layer, col] = count

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(heatmap, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(fn_names)))
ax.set_xticklabels(fn_names, rotation=45, ha='right')
ax.set_ylabel('Layer')
ax.set_xlabel('Best-fit Function')
ax.set_title(f'What Each Layer Learned \u2014 {GENOME.title()} KAN\n'
             '(count of edges best-fit to each function type)')
plt.colorbar(im, label='Edge count')
plt.tight_layout()
plt.show()

# Summary statistics
print(f"\nFunction distribution across {len(all_edges_full)} edges:")
from collections import Counter
fn_dist = Counter(e['best_match']['name'] for e in all_edges_full)
for fn, count in fn_dist.most_common():
    pct = 100 * count / len(all_edges_full)
    print(f"  {fn:12s}: {count:4d} ({pct:.1f}%)")

## 3. What Did the Splines Learn? (Visual + Numerical)

For the most visually impactful spline edges, we show:
- The learned spline shape
- Its closest known function + residual
- How sweeping this parameter changes the output image

In [ ]:
from src.kan import FlattenKANParameters
from src.visualize import discover_interesting_kan_sweeps, sweep_weight, plot_sweep_grid

kan_flat = FlattenKANParameters(kan_model)

# Find most impactful parameters
interesting = discover_interesting_kan_sweeps(kan_model, kan_flat, target_img,
                                              n_candidates_per_group=8, top_k=6)

# Multi-panel figure: spline shape | best fit + residual | image sweep
fig = plt.figure(figsize=(20, 4 * len(interesting)))
gs = gridspec.GridSpec(len(interesting), 3, width_ratios=[1, 1, 3])

params = kan_flat.flatten()

for row, entry in enumerate(interesting):
    flat_idx = entry['flat_idx']
    desc = entry['description']

    # Parse layer/in/out from description like "Layer 5 coeffs[3,7,12]"
    # Find the corresponding edge info
    from src.visualize import get_kan_param_info
    info = get_kan_param_info(kan_model, flat_idx)
    layer_idx = info['layer_idx']
    layer = kan_model.layers[layer_idx]

    # Extract spline for this edge (use first two indices for out, in)
    indices = info['local_shape_indices']
    if info['param_type'] == 'coeffs' and len(indices) >= 2:
        out_idx, in_idx = indices[0], indices[1]
        raw_inputs, spline_values, _ = extract_spline_curve(layer, in_idx, out_idx)
        match = fit_known_function(raw_inputs, spline_values)

        # Panel 1: Spline shape
        ax1 = fig.add_subplot(gs[row, 0])
        ax1.plot(raw_inputs, spline_values, 'b-', linewidth=2)
        ax1.set_title(f'{desc}\nSpline Shape', fontsize=9)
        ax1.grid(True, alpha=0.3)

        # Panel 2: Best fit + residual
        ax2 = fig.add_subplot(gs[row, 1])
        if match['fitted_curve'] is not None:
            ax2.plot(raw_inputs, match['fitted_curve'], 'r-', label=f"{match['name']}", linewidth=1.5)
            residual = spline_values - match['fitted_curve']
            ax2.plot(raw_inputs, residual, 'g--', label='Residual', linewidth=1, alpha=0.7)
        ax2.set_title(f"\u2248 {match['name']} (L2={match['l2_distance']:.3f})", fontsize=9)
        ax2.legend(fontsize=7)
        ax2.grid(True, alpha=0.3)
    else:
        ax1 = fig.add_subplot(gs[row, 0])
        ax1.text(0.5, 0.5, f'{desc}\n(not a spline coeff)', ha='center', va='center')
        ax2 = fig.add_subplot(gs[row, 1])

    # Panel 3: Image sweep strip
    ax3 = fig.add_subplot(gs[row, 2])
    sweep_imgs = sweep_weight(params, flat_idx, kan_flat, img_size=128, n=7)
    sweep_np = sweep_imgs.detach().cpu().numpy()
    strip = np.concatenate(sweep_np, axis=1)
    ax3.imshow(strip.clip(0, 1))
    ax3.set_title(f'Weight Sweep: {desc}', fontsize=9)
    ax3.set_xticks([]); ax3.set_yticks([])

plt.suptitle(f'Spline Analysis \u2014 {GENOME.title()} KAN', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Can This Scale to Text?

### Theoretical Analysis

CPPNs are fundamentally **coordinate-based function approximators**: they map spatial coordinates (x, y) to output values (color). This is naturally suited to images because pixels have spatial structure — nearby pixels should have similar values.

**Key constraints for text:**
- Text is sequential, not spatial. There's no 2D grid.
- Text has discrete tokens, not continuous color values.
- Long-range dependencies in text span the entire sequence (not just local neighborhoods).

**What would need to change:**
1. **Input encoding**: Replace (x, y, d, bias) with (position, bias) or learned positional embeddings
2. **Output head**: Replace HSV → RGB with logits → softmax for token prediction
3. **Architecture depth**: Text likely needs deeper networks or attention for long-range deps
4. **Training signal**: Cross-entropy loss instead of MSE on pixel values

**Why it might partially work:**
- KAN's learnable activation functions could capture complex position→token mappings
- The CPPN's ability to generate patterns from coordinates is analogous to positional encoding
- Evolutionary search (memetic) could explore discrete loss landscapes better than pure SGD

**Why it probably won't scale well:**
- CPPNs have no mechanism for attending to other positions (no self-attention)
- Each position is processed independently — there's no information flow between positions
- Language requires modeling dependencies between tokens, not just position → token mapping

### Toy Prototype

Below we test whether a KAN-CPPN can learn simple 1D signals (sine, square wave, etc.) as a sanity check. If it can't even learn periodic 1D functions, text is out of the question.

In [ ]:
test_signals = make_test_signals(seq_len=200)

fig, axes = plt.subplots(len(test_signals), 2, figsize=(14, 3 * len(test_signals)))

for i, (name, (positions, target)) in enumerate(test_signals.items()):
    model = SequenceKAN(n_layers=4, hidden_size=16, output_size=1)
    losses = train_sequence_kan(model, target, positions, n_iters=3000, lr=3e-3)

    # Generate learned signal
    _, learned = model.generate_signal(seq_len=200)

    # Plot signal comparison
    ax = axes[i, 0]
    ax.plot(positions.numpy(), target[:, 0].numpy(), 'b-', label='Target', linewidth=2)
    ax.plot(positions.numpy(), learned[:, 0].detach().numpy(), 'r--', label='KAN output', linewidth=1.5)
    ax.set_title(f'{name} \u2014 Final MSE: {losses[-1]:.6f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Plot loss curve
    ax = axes[i, 1]
    ax.plot(losses)
    ax.set_yscale('log')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('MSE')
    ax.set_title(f'{name} \u2014 Loss Curve')
    ax.grid(True, alpha=0.3)

plt.suptitle('1D Signal Approximation with KAN-CPPN', fontsize=14)
plt.tight_layout()
plt.show()

print("\nConclusion: KAN-CPPNs can/cannot learn 1D periodic signals,")
print("suggesting they are/aren't suitable building blocks for sequential tasks.")
print("The fundamental limitation is the lack of inter-position information flow.")

## 5. How Swarm & Memetic KAN Actually Work

### Spline Type: Linear Interpolation (Piecewise Linear)

The KAN layers use **piecewise linear splines** — NOT B-splines, quadratic, or cubic.

Each edge (input_i → output_j) has:
- A **fixed grid** of 20 knots at `linspace(0, 1, 20)`
- **20 learnable coefficients** (one per knot)
- **Linear interpolation** between adjacent knots

The input is mapped to [0, 1] via sigmoid, then the two nearest knots are found and linearly interpolated. This is the simplest possible spline — computationally cheap and fully differentiable.

**Full edge computation:**
```
spline_ij(x) = weight_ij * lerp(coeffs[k], coeffs[k+1], frac)
```
where `k = floor(sigmoid(x) * 19)` and `frac` is the fractional part.

### SwarmKAN: PSO + SGD Hybrid

**Particle Swarm Optimization** on spline coefficients, interleaved with gradient descent.

**Algorithm per training step:**
1. Standard SGD step (Adam optimizer, normalized gradients)
2. Every 5 SGD steps, perform PSO update:
   - Each layer maintains 5 "particles" (alternative coefficient vectors)
   - Particle velocities updated via classic PSO formula:
     `v = 0.7*v + 1.5*r1*(personal_best - pos) + 1.5*r2*(global_best - pos)`
   - Particle positions updated: `pos += v`
   - Active coefficients blended with particle 0: `coeffs = 0.9*coeffs + 0.1*particle_0`

**Why this design:**
- SGD handles local optimization (fast convergence in the loss basin)
- PSO explores the activation function landscape (particles try different spline shapes)
- Soft blending (10%) prevents PSO from destroying SGD's progress
- Each layer has independent particle populations (no cross-layer interference)

### MemeticKAN: NES + SGD Hybrid

**Natural Evolution Strategy** (OpenAI-ES style) for global search, plus SGD for local refinement.

**Algorithm per generation:**
1. **ES gradient estimation:**
   - Take current spline params as "center" (base_weight is frozen/excluded)
   - Sample 20 random perturbation vectors ε
   - For each ε, evaluate fitness at (center + σε) and (center - σε) [antithetic pairs]
   - ES gradient = Σ [(f+ - f-) / (|f+| + |f-| + ε)] * ε / (2 * pop_size * σ)
   - Apply ES gradient to center params (only if it improves fitness)

2. **SGD local refinement:**
   - Create fresh Adam optimizer (old one's state is invalid after ES jump)
   - Run 50-100 SGD steps with normalized gradients
   - This refines the ES's coarse global move

**Key difference from SwarmKAN:**
- SwarmKAN perturbs per-layer independently; MemeticKAN perturbs globally
- SwarmKAN maintains persistent particle population; MemeticKAN samples fresh each generation
- MemeticKAN uses antithetic sampling for variance reduction (stronger signal)
- MemeticKAN gates ES updates (only accepts improvements)

In [ ]:
# Visualize the PSO and NES algorithms step by step

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- PSO Visualization ---
ax = axes[0]
ax.set_title('SwarmKAN: PSO on Spline Coefficients', fontsize=12)

# Show a 2D slice of the coefficient space with particle trajectories
np.random.seed(42)
n_steps = 20
n_particles = 5

# Simulate PSO in 2D for illustration
positions = np.random.randn(n_particles, 2) * 0.5
velocities = np.zeros_like(positions)
personal_best = positions.copy()
personal_best_scores = np.full(n_particles, np.inf)
global_best = positions[0].copy()
global_best_score = np.inf

# Simple 2D loss landscape: (x-0.3)^2 + (y+0.2)^2
def loss_2d(pos):
    return (pos[0] - 0.3)**2 + (pos[1] + 0.2)**2

trajectories = [[] for _ in range(n_particles)]
for step in range(n_steps):
    for i in range(n_particles):
        trajectories[i].append(positions[i].copy())
        score = loss_2d(positions[i])
        if score < personal_best_scores[i]:
            personal_best_scores[i] = score
            personal_best[i] = positions[i].copy()
        if score < global_best_score:
            global_best_score = score
            global_best = positions[i].copy()

    r1 = np.random.rand(n_particles, 2)
    r2 = np.random.rand(n_particles, 2)
    velocities = 0.7 * velocities + 1.5 * r1 * (personal_best - positions) + 1.5 * r2 * (global_best - positions)
    positions += velocities

particle_colors = plt.cm.Set1(np.linspace(0, 1, n_particles))
for i in range(n_particles):
    traj = np.array(trajectories[i])
    ax.plot(traj[:, 0], traj[:, 1], '-o', color=particle_colors[i], markersize=3,
            alpha=0.6, label=f'Particle {i}')
    ax.plot(traj[0, 0], traj[0, 1], 's', color=particle_colors[i], markersize=8)
    ax.plot(traj[-1, 0], traj[-1, 1], '*', color=particle_colors[i], markersize=12)

ax.plot(0.3, -0.2, 'kx', markersize=15, markeredgewidth=3, label='Optimum')
ax.legend(fontsize=7, loc='upper right')
ax.set_xlabel('Coefficient dimension 1')
ax.set_ylabel('Coefficient dimension 2')
ax.grid(True, alpha=0.3)

# --- NES Visualization ---
ax = axes[1]
ax.set_title('MemeticKAN: NES Gradient Estimation', fontsize=12)

center = np.array([0.0, 0.0])
sigma = 0.3
n_perturbations = 8

np.random.seed(123)
for gen in range(3):
    color = plt.cm.Blues(0.4 + gen * 0.2)
    for i in range(n_perturbations):
        eps = np.random.randn(2) * sigma
        pos = center + eps
        neg = center - eps
        ax.plot([neg[0], pos[0]], [neg[1], pos[1]], '-', color=color, alpha=0.3, linewidth=1)
        ax.plot(pos[0], pos[1], 'o', color='green', markersize=4, alpha=0.5)
        ax.plot(neg[0], neg[1], 'o', color='red', markersize=4, alpha=0.5)

    ax.plot(center[0], center[1], 'D', color=color, markersize=10,
            label=f'Center (gen {gen})')
    # Simulate ES gradient move
    center = center + np.array([0.1, -0.07]) * (gen + 1) * 0.5

ax.plot(0.3, -0.2, 'kx', markersize=15, markeredgewidth=3, label='Optimum')
ax.legend(fontsize=8)
ax.set_xlabel('Parameter dimension 1')
ax.set_ylabel('Parameter dimension 2')
ax.grid(True, alpha=0.3)

plt.suptitle('Algorithm Visualization (2D Projection)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Design Choices That Mattered (Transferable Insights)

These are the inductive biases and engineering decisions that made the KAN-CPPN work. Each one is a transferable "skill" for future deep learning projects.

In [ ]:
# Ablation study: test key design choices

from src.kan import KAN_CPPN, KANCPPNLayer

cfg = GENOME_CONFIGS[GENOME]

def train_and_measure(model, target, n_iters=2000):
    """Quick training run, return final loss."""
    from src.train import train_sgd
    losses, _ = train_sgd(model, target, lr=3e-3, n_iters=n_iters, log_interval=0)
    return losses

# --- Ablation 1: Orthogonal vs. Kaiming init ---
torch.manual_seed(42)
kan_ortho = KAN_CPPN(n_layers=cfg['n_layers'], hidden_size=cfg['hidden_size'])
# Default is orthogonal

torch.manual_seed(42)
kan_kaiming = KAN_CPPN(n_layers=cfg['n_layers'], hidden_size=cfg['hidden_size'])
# Override with kaiming
for layer in kan_kaiming.layers:
    torch.nn.init.kaiming_uniform_(layer.base_weight)

losses_ortho = train_and_measure(kan_ortho, target_img)
losses_kaiming = train_and_measure(kan_kaiming, target_img)

# --- Ablation 2: With vs. without residual base path ---
# (We can't easily remove base_weight without modifying the layer,
#  so we zero it out instead)
torch.manual_seed(42)
kan_no_residual = KAN_CPPN(n_layers=cfg['n_layers'], hidden_size=cfg['hidden_size'])
with torch.no_grad():
    for layer in kan_no_residual.layers:
        layer.base_weight.fill_(0)
        layer.base_weight.requires_grad_(False)

losses_no_residual = train_and_measure(kan_no_residual, target_img)

# --- Signal propagation test ---
torch.manual_seed(42)
kan_test = KAN_CPPN(n_layers=cfg['n_layers'], hidden_size=cfg['hidden_size'])
test_input = torch.randn(100, 4)  # Random batch

with torch.no_grad():
    x = test_input
    stds_ortho = [x.std().item()]
    for layer in kan_test.layers:
        x = layer(x)
        stds_ortho.append(x.std().item())

# Reset with kaiming
for layer in kan_test.layers:
    torch.nn.init.kaiming_uniform_(layer.base_weight)

with torch.no_grad():
    x = test_input
    stds_kaiming = [x.std().item()]
    for layer in kan_test.layers:
        x = layer(x)
        stds_kaiming.append(x.std().item())

# --- Plot ablation results ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Convergence comparison
ax = axes[0]
ax.plot(losses_ortho, label='Orthogonal init (default)', linewidth=1.5)
ax.plot(losses_kaiming, label='Kaiming init', linewidth=1.5)
ax.plot(losses_no_residual, label='No residual path', linewidth=1.5)
ax.set_yscale('log')
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE')
ax.set_title('Ablation: Init & Residual Path')
ax.legend()
ax.grid(True, alpha=0.3)

# Signal propagation
ax = axes[1]
ax.plot(stds_ortho, 'o-', label='Orthogonal', linewidth=1.5)
ax.plot(stds_kaiming, 's-', label='Kaiming', linewidth=1.5)
ax.set_xlabel('Layer')
ax.set_ylabel('Activation Std')
ax.set_title('Signal Propagation Through Layers')
ax.legend()
ax.grid(True, alpha=0.3)

# Summary table
ax = axes[2]
ax.axis('off')
table_data = [
    ['Design Choice', 'Effect', 'When to Use'],
    ['Orthogonal base_weight', 'Preserves signal norm\nthrough deep nets', 'Deep networks (>10 layers)\nwith residual paths'],
    ['Residual base + spline', 'Prevents signal collapse,\nspline adds flexibility', 'Any learnable-activation\narchitecture'],
    ['Sigmoid grid normalization', 'Maps inputs to valid\ngrid domain [0,1]', 'Spline-based networks\nwith fixed grids'],
    ['Gradient normalization', 'Decouples lr from\ngradient magnitude', 'Networks with varying\ngradient scales'],
    ['Exclude base_weight\nfrom ES', 'Preserves orthogonality\nduring evolution', 'Hybrid evolutionary +\ngradient methods'],
    ['Antithetic sampling', 'Halves ES gradient\nvariance', 'Any evolution strategy\nestimator'],
    ['Fresh optimizer\nper generation', 'Prevents stale Adam\nstate after ES jump', 'Memetic algorithms\nwith optimizer resets'],
]

table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                  loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.8)
ax.set_title('Transferable Design Insights', fontsize=12, pad=20)

plt.tight_layout()
plt.show()